# 2 Assets

In [1]:
using YFinance
using DataFrames
using Statistics
using LinearAlgebra
using PlotlyLight
using DelimitedFiles

In [2]:
stocks = ["AAPL", "TSLA"]

stock_data = [
    YFinance.get_prices(
        ticker,
        startdt="2015-01-01",
        enddt="2026-01-01",
        interval="1d"
    )
    for ticker in stocks
] 

2-element Vector{OrderedCollections.OrderedDict{String, Union{Vector{Float64}, Vector{Dates.DateTime}, String}}}:
 OrderedCollections.OrderedDict("ticker" => "AAPL", "timestamp" => [Dates.DateTime("2015-01-02T14:30:00"), Dates.DateTime("2015-01-05T14:30:00"), Dates.DateTime("2015-01-06T14:30:00"), Dates.DateTime("2015-01-07T14:30:00"), Dates.DateTime("2015-01-08T14:30:00"), Dates.DateTime("2015-01-09T14:30:00"), Dates.DateTime("2015-01-12T14:30:00"), Dates.DateTime("2015-01-13T14:30:00"), Dates.DateTime("2015-01-14T14:30:00"), Dates.DateTime("2015-01-15T14:30:00")  …  Dates.DateTime("2025-12-17T14:30:00"), Dates.DateTime("2025-12-18T14:30:00"), Dates.DateTime("2025-12-19T14:30:00"), Dates.DateTime("2025-12-22T14:30:00"), Dates.DateTime("2025-12-23T14:30:00"), Dates.DateTime("2025-12-24T14:30:00"), Dates.DateTime("2025-12-26T14:30:00"), Dates.DateTime("2025-12-29T14:30:00"), Dates.DateTime("2025-12-30T14:30:00"), Dates.DateTime("2025-12-31T14:30:00")], "open" => [24.62719552301581, 23.9

In [3]:
prices = DataFrame(
    Date = stock_data[1]["timestamp"],
    AAPL = stock_data[1]["adjclose"],
    TSLA = stock_data[2]["adjclose"]
)

Row,Date,AAPL,TSLA
,DateTime,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207
2,2015-01-05T14:30:00,23.4908,14.006
3,2015-01-06T14:30:00,23.493,14.0853
4,2015-01-07T14:30:00,23.8224,14.0633
5,2015-01-08T14:30:00,24.7377,14.0413
6,2015-01-09T14:30:00,24.7643,13.7773
7,2015-01-12T14:30:00,24.1541,13.4807
8,2015-01-13T14:30:00,24.3685,13.6167
9,2015-01-14T14:30:00,24.2757,12.846


In [4]:
first(prices, 5) # displays first 5 rows

Row,Date,AAPL,TSLA
,DateTime,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207
2,2015-01-05T14:30:00,23.4908,14.006
3,2015-01-06T14:30:00,23.493,14.0853
4,2015-01-07T14:30:00,23.8224,14.0633
5,2015-01-08T14:30:00,24.7377,14.0413


In [5]:
println(size(prices)) # number rows and columns

(2766, 3)


In [6]:
[sum(ismissing, col) for col in eachcol(prices)] # check for missing values

3-element Vector{Int64}:
 0
 0
 0

## Logarithmic Daily Returns

In [7]:
price_matrix = Matrix(prices[:, [:AAPL, :TSLA]])

returns_matrix = log.(price_matrix[2:end, :] ./ price_matrix[1:end-1, :])

returns = DataFrame(returns_matrix, [:AAPL, :TSLA])

Row,AAPL,TSLA
,Float64,Float64
1,-0.0285759,-0.0429503
2,9.43448e-5,0.00564826
3,0.0139246,-0.00156315
4,0.0377025,-0.00156553
5,0.00107216,-0.0189806
6,-0.0249497,-0.0217681
7,0.00883975,0.0100379
8,-0.00381759,-0.0582621
9,-0.0275156,-0.00426461


## Sample Covariance Matrix

In [8]:
mean_returns = mean(Matrix(returns), dims=1)

1×2 Matrix{Float64}:
 0.00087428  0.00123913

In [9]:
centered_returns = Matrix(returns) .- mean_returns

2765×2 Matrix{Float64}:
 -0.0294502    -0.0441895
 -0.000779935   0.00440913
  0.0130503    -0.00280228
  0.0368282    -0.00280466
  0.000197876  -0.0202198
 -0.0258239    -0.0230073
  0.00796547    0.00879882
 -0.00469187   -0.0595012
 -0.0283899    -0.00550374
 -0.00867452    0.00499562
  0.0245566    -0.00716124
  0.00673115    0.0226488
  0.0248084     0.024127
  ⋮            
 -0.0159726     0.033765
  0.000948197   0.0289541
 -0.0110125    -0.0485137
  0.000412564   0.0326574
  0.00454837   -0.00573851
 -0.0107892     0.0142881
  0.00424234   -0.00774648
  0.00443537   -0.00156871
 -0.00237263   -0.0224977
  0.000441555  -0.0345103
 -0.00336144   -0.0126389
 -0.0053518    -0.0116578

In [10]:
n = size(returns, 1)
cov_manual = centered_returns' * centered_returns / (n - 1)
print(cov_manual)

[0.0003295796777199375 0.00028647727366398854; 0.00028647727366398854 0.0013158307114027078]

In [11]:
cov_julia = cov(Matrix(returns))
print(cov_julia)

[0.0003295796777199375 0.0002864772736639885; 0.0002864772736639885 0.0013158307114027078]

In [12]:
println(isapprox(cov_manual, cov_julia))

true


## Sample Correlation Matrix

In [13]:
std_returns = std(Matrix(returns), dims=1) # standard deviation

1×2 Matrix{Float64}:
 0.0181543  0.0362744

In [14]:
corr_manual = cov_manual ./ (std_returns' * std_returns)
println(corr_manual)

[0.9999999999999997 0.4350207304502293; 0.4350207304502293 0.9999999999999999]


In [15]:
corr_julia = cor(Matrix(returns))
println(corr_julia)

[1.0 0.4350207304502293; 0.4350207304502293 1.0]


In [16]:
println(isapprox(corr_manual, corr_julia))

true


## Verifying Mathematical properties

In [17]:
println(isapprox(cov_julia, cov_julia'))
println(isapprox(corr_julia, corr_julia'))

true
true


They are both symmetric

In [18]:
println(eigvals(cov_julia))
println(eigvals(corr_julia))

[0.00025240527023550355, 0.0013930051188871418]
[0.5649792695497707, 1.4350207304502294]


Valid, the eigenvalues are non-negative and within numerical tolerance

In [19]:
println(sum(eigvals(corr_julia)))

2.0


In [20]:
eigenvectors = eigen(Symmetric(corr_julia)).vectors
println(eigenvectors)

[-0.7071067811865475 0.7071067811865475; 0.7071067811865475 0.7071067811865475]


Sum of the eigenvalues of the corelation matrix equals the number of assets.

In [21]:
println(minimum(eigvals(Symmetric(cov_julia))) >= -1e-10)
println(minimum(eigvals(Symmetric(corr_julia))) >= -1e-10)

true
true


In [22]:
println(all(corr_julia .>= -1))
println(all(corr_julia .<= 1))

true
true


Pearson correlation coefficent is between -1 and +1 as required

## Visuals

In [23]:
p = PlotlyLight.Plot()
p.data = [
    PlotlyLight.Config(
        type="heatmap",
        z=corr_julia,
        x=["AAPL", "TSLA"],
        y=["AAPL", "TSLA"],
        colorscale=[[0, "white"], [0.5, "lightpink"], [1, "deeppink"]],
        zmin=-1,
        zmax=1,
        text=round.(corr_julia, digits=3),
        texttemplate="%{text}"
    )
]
p.layout.title = "Apple and Tesla Correlation Matrix"
p.layout.yaxis.autorange = "reversed"
p            

PlotlyLight.jl Plot

In [24]:
x = returns.AAPL
y = returns.TSLA
# Calculate OLS regression line
slope = cov(x, y) / var(x)
intercept = mean(y) - slope * mean(x)
r = cor(x, y)

p = PlotlyLight.Plot()

p.data = [
    PlotlyLight.Config(
        type="scatter",
        x=x,
        y=y,
        mode="markers",
        marker=PlotlyLight.Config(color="hotpink"),
        name="Daily Returns"
    ),
    PlotlyLight.Config(
        type="scatter",
        x=[minimum(x), maximum(x)],
        y=intercept .+ slope .* [minimum(x), maximum(x)],
        mode="lines",
        line=PlotlyLight.Config(color="purple", width=3),
        name="OLS Trendline"
    )
]

p.layout.title = "Apple vs Tesla Daily Returns (r = $(round(r, digits=3)))"
p.layout.xaxis.title = "Apple Daily Log Returns"
p.layout.yaxis.title = "Tesla Daily Log Returns"

p

PlotlyLight.jl Plot

In [25]:
p = PlotlyLight.Plot()

p.data = [
    PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.AAPL),
        y=returns.AAPL,
        mode="lines",
        name="Apple",
        line=PlotlyLight.Config(color="hotpink")
    ),
    PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.TSLA),
        y=returns.TSLA,
        mode="lines",
        name="Tesla",
        line=PlotlyLight.Config(color="royalblue")
    )
]

p.layout.title = "Apple and Tesla Daily Log Returns"
p.layout.xaxis.title = "Trading Day"
p.layout.yaxis.title = "Daily Log Return"

p

PlotlyLight.jl Plot

In [26]:
writedlm("julia_cov_2.csv", cov_julia, ',')
writedlm("julia_corr_2.csv", corr_julia, ',')

# 5 Assets

In [27]:
stocks = ["AAPL", "TSLA", "MSFT", "KO", "JNJ"]

stock_data = [
    YFinance.get_prices(
        ticker,
        startdt="2015-01-01",
        enddt="2026-01-01",
        interval="1d"
    )
    for ticker in stocks
] 

5-element Vector{OrderedCollections.OrderedDict{String, Union{Vector{Float64}, Vector{Dates.DateTime}, String}}}:
 OrderedCollections.OrderedDict("ticker" => "AAPL", "timestamp" => [Dates.DateTime("2015-01-02T14:30:00"), Dates.DateTime("2015-01-05T14:30:00"), Dates.DateTime("2015-01-06T14:30:00"), Dates.DateTime("2015-01-07T14:30:00"), Dates.DateTime("2015-01-08T14:30:00"), Dates.DateTime("2015-01-09T14:30:00"), Dates.DateTime("2015-01-12T14:30:00"), Dates.DateTime("2015-01-13T14:30:00"), Dates.DateTime("2015-01-14T14:30:00"), Dates.DateTime("2015-01-15T14:30:00")  …  Dates.DateTime("2025-12-17T14:30:00"), Dates.DateTime("2025-12-18T14:30:00"), Dates.DateTime("2025-12-19T14:30:00"), Dates.DateTime("2025-12-22T14:30:00"), Dates.DateTime("2025-12-23T14:30:00"), Dates.DateTime("2025-12-24T14:30:00"), Dates.DateTime("2025-12-26T14:30:00"), Dates.DateTime("2025-12-29T14:30:00"), Dates.DateTime("2025-12-30T14:30:00"), Dates.DateTime("2025-12-31T14:30:00")], "open" => [24.627207182737358, 23.

In [28]:
prices = DataFrame(
    Date = stock_data[1]["timestamp"],
    AAPL = stock_data[1]["adjclose"],
    TSLA = stock_data[2]["adjclose"],
    MSFT = stock_data[3]["adjclose"],
    KO = stock_data[4]["adjclose"],
    JNJ = stock_data[5]["adjclose"]
)

Row,Date,AAPL,TSLA,MSFT,KO,JNJ
,DateTime,Float64,Float64,Float64,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207,39.6072,29.2159,75.7374
2,2015-01-05T14:30:00,23.4908,14.006,39.243,29.2159,75.2084
3,2015-01-06T14:30:00,23.493,14.0853,38.667,29.4378,74.8389
4,2015-01-07T14:30:00,23.8224,14.0633,39.1582,29.8052,76.491
5,2015-01-08T14:30:00,24.7377,14.0413,40.3102,30.1658,77.0924
6,2015-01-09T14:30:00,24.7643,13.7773,39.9714,29.833,76.0417
7,2015-01-12T14:30:00,24.1541,13.4807,39.4717,29.5626,75.7808
8,2015-01-13T14:30:00,24.3685,13.6167,39.2684,29.5556,75.9113
9,2015-01-14T14:30:00,24.2757,12.846,38.9296,29.5071,75.3606


In [29]:
first(prices, 5) # displays first 5 rows

Row,Date,AAPL,TSLA,MSFT,KO,JNJ
,DateTime,Float64,Float64,Float64,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207,39.6072,29.2159,75.7374
2,2015-01-05T14:30:00,23.4908,14.006,39.243,29.2159,75.2084
3,2015-01-06T14:30:00,23.493,14.0853,38.667,29.4378,74.8389
4,2015-01-07T14:30:00,23.8224,14.0633,39.1582,29.8052,76.491
5,2015-01-08T14:30:00,24.7377,14.0413,40.3102,30.1658,77.0924


In [30]:
println(size(prices)) # number rows and columns

(2766, 6)


In [31]:
[sum(ismissing, col) for col in eachcol(prices)] # check for missing values

6-element Vector{Int64}:
 0
 0
 0
 0
 0
 0

## Logarithmic Daily Returns

In [32]:
price_matrix = Matrix(prices[:, [:AAPL, :TSLA, :MSFT, :KO, :JNJ]])

returns_matrix = log.(price_matrix[2:end, :] ./ price_matrix[1:end-1, :])

returns = DataFrame(returns_matrix, [:AAPL, :TSLA, :MSFT, :KO, :JNJ])

Row,AAPL,TSLA,MSFT,KO,JNJ
,Float64,Float64,Float64,Float64,Float64
1,-0.0285759,-0.0429503,-0.00923854,0.0,-0.00700886
2,9.38576e-5,0.00564826,-0.0147855,0.00756534,-0.00492522
3,0.0139246,-0.00156315,0.0126248,0.012405,0.0218352
4,0.0377027,-0.00156553,0.0289939,0.0120233,0.00783204
5,0.00107208,-0.0189806,-0.00844078,-0.0110936,-0.0137231
6,-0.0249495,-0.0217681,-0.0125813,-0.00910497,-0.00343624
7,0.00883959,0.0100379,-0.00516337,-0.000234232,0.0017199
8,-0.00381798,-0.0582621,-0.00866579,-0.00164323,-0.00728113
9,-0.0275154,-0.00426461,-0.0104987,-0.00423845,-0.0146259


## Sample Covariance Matrix

In [33]:
mean_returns = mean(Matrix(returns), dims=1)

1×5 Matrix{Float64}:
 0.00087428  0.00123913  0.0009027  0.000308592  0.000357775

In [34]:
centered_returns = Matrix(returns) .- mean_returns

2765×5 Matrix{Float64}:
 -0.0294502    -0.0441895   -0.0101412    -0.000308592  -0.00736663
 -0.000780422   0.00440913  -0.0156882     0.00725675   -0.00528299
  0.0130503    -0.00280228   0.0117221     0.0120964     0.0214774
  0.0368284    -0.00280466   0.0280912     0.0117147     0.00747427
  0.000197799  -0.0202198   -0.00934348   -0.0114021    -0.0140809
 -0.0258238    -0.0230073   -0.013484     -0.00941356   -0.00379401
  0.00796531    0.00879882  -0.00606607   -0.000542824   0.00136212
 -0.00469226   -0.0595012   -0.00956849   -0.00195182   -0.0076389
 -0.0283896    -0.00550374  -0.0114014    -0.00454704   -0.0149836
 -0.00867485    0.00499562   0.0156697     0.00322483    0.0146523
  0.024557     -0.00716124   0.00233613    0.0143958    -0.0271454
  0.00673092    0.0226488   -0.0110859     0.00431479    0.00564645
  0.0248087     0.024127     0.0251062     0.00933084    0.0177305
  ⋮                                                     
 -0.0159726     0.033765    -0.00868589   

In [35]:
n = size(returns, 1)
cov_manual = centered_returns' * centered_returns / (n - 1)
print(cov_manual)

[0.00032957961604200087 0.000286477444067879 0.00020422549733443854 7.013550350885698e-5 6.598757869627009e-5; 0.000286477444067879 0.0013158307114027078 0.0002520129120124958 5.683312610610064e-5 3.967536988255415e-5; 0.00020422549733443854 0.0002520129120124958 0.0002863342229928084 6.881317674954144e-5 6.486782589086453e-5; 7.013550350885698e-5 5.683312610610064e-5 6.881317674954144e-5 0.00012611933786777898 6.481013438844835e-5; 6.598757869627009e-5 3.967536988255415e-5 6.486782589086453e-5 6.481013438844835e-5 0.0001318160608288322]

In [36]:
cov_julia = cov(Matrix(returns))
print(cov_julia)

[0.0003295796160420008 0.0002864774440678789 0.00020422549733443851 7.013550350885698e-5 6.598757869627009e-5; 0.0002864774440678789 0.0013158307114027078 0.00025201291201249577 5.6833126106100635e-5 3.9675369882554146e-5; 0.00020422549733443851 0.00025201291201249577 0.00028633422299280837 6.881317674954143e-5 6.486782589086453e-5; 7.013550350885698e-5 5.6833126106100635e-5 6.881317674954143e-5 0.00012611933786777895 6.481013438844834e-5; 6.598757869627009e-5 3.9675369882554146e-5 6.486782589086453e-5 6.481013438844834e-5 0.0001318160608288322]

In [37]:
println(isapprox(cov_manual, cov_julia))

true


## Sample Correlation Matrix

In [38]:
std_returns = std(Matrix(returns), dims=1) # standard deviation

1×5 Matrix{Float64}:
 0.0181543  0.0362744  0.0169214  0.0112303  0.0114811

In [39]:
corr_manual = cov_manual ./ (std_returns' * std_returns)
println(corr_manual)

[1.0 0.43502102991668623 0.6648032920346455 0.34400670420393153 0.3165904838978059; 0.43502102991668623 0.9999999999999999 0.4105689992763037 0.13951172055560468 0.09526575428258095; 0.6648032920346455 0.4105689992763037 1.0000000000000007 0.3621130173008559 0.3338939489614802; 0.34400670420393153 0.13951172055560468 0.3621130173008559 1.0000000000000002 0.502652580465893; 0.3165904838978059 0.09526575428258095 0.3338939489614802 0.502652580465893 1.0000000000000002]


In [40]:
corr_julia = cor(Matrix(returns))
println(corr_julia)

[1.0 0.4350210299166862 0.6648032920346454 0.3440067042039315 0.3165904838978059; 0.4350210299166862 1.0 0.4105689992763036 0.13951172055560465 0.09526575428258094; 0.6648032920346454 0.4105689992763036 1.0 0.3621130173008557 0.33389394896148017; 0.3440067042039315 0.13951172055560465 0.3621130173008557 1.0 0.5026525804658928; 0.3165904838978059 0.09526575428258094 0.33389394896148017 0.5026525804658928 1.0]


In [41]:
println(isapprox(corr_manual, corr_julia))

true


## Verifying Mathematical Properties

In [42]:
println(isapprox(cov_julia, cov_julia'))
println(isapprox(corr_julia, corr_julia'))

true
true


In [43]:
println(eigvals(cov_julia))
println(eigvals(corr_julia))

[6.38846123416052e-5, 0.00010186356346220908, 0.00013996370171451736, 0.0004055119703668266, 0.0014784561012489705]
[0.33425710632943956, 0.495244039637282, 0.58271814245994, 1.104083564245457, 2.48369714732788]


In [44]:
println(sum(eigvals(corr_julia)))

4.999999999999998


Sum of the eigenvalues of the correlation matrix is equal to the number of assets.

In [45]:
println(minimum(eigvals(Symmetric(cov_julia))) >= -1e-10)
println(minimum(eigvals(Symmetric(corr_julia))) >= -1e-10)

true
true


In [46]:
eigenvectors = eigen(Symmetric(corr_julia)).vectors
println(eigenvectors)

[0.7131343339998002 -0.004521644938528635 0.4051265186078986 -0.2385874718033432 -0.5199687477309036; -0.038606891113310525 0.08315915138197161 -0.731228795658712 -0.5713088795899552 -0.36125430931848407; -0.6997513213637262 -0.020973597812134204 0.44562170535021917 -0.19572583841708951 -0.5225140077942637; 0.01653342392555976 -0.7054560043242447 -0.2715144879415847 0.5061064397350524 -0.414963403630606; 0.004864542346749365 0.7035311254139124 -0.16993566688230827 0.5676529311426098 -0.39231659496898613]


In [47]:
eigenvectors_df = DataFrame(eigenvectors, :auto)

rename!(eigenvectors_df, [
    "Eigenvector 1", "Eigenvector 2",
    "Eigenvector 3", "Eigenvector 4", "Eigenvector 5"
])

eigenvectors_df.Stock = ["AAPL", "TSLA", "MSFT", "KO", "JNJ"]

select!(eigenvectors_df, :Stock, :)

eigenvectors_df

Row,Stock,Eigenvector 1,Eigenvector 2,Eigenvector 3,Eigenvector 4,Eigenvector 5
,String,Float64,Float64,Float64,Float64,Float64
1,AAPL,0.713134,-0.00452164,0.405127,-0.238587,-0.519969
2,TSLA,-0.0386069,0.0831592,-0.731229,-0.571309,-0.361254
3,MSFT,-0.699751,-0.0209736,0.445622,-0.195726,-0.522514
4,KO,0.0165334,-0.705456,-0.271514,0.506106,-0.414963
5,JNJ,0.00486454,0.703531,-0.169936,0.567653,-0.392317


In [48]:
println(all(corr_julia .>= -1))
println(all(corr_julia .<= 1))

true
true


## Visuals

In [49]:
p = PlotlyLight.Plot()
p.data = [
    PlotlyLight.Config(
        type="heatmap",
        z=corr_julia,
        x=stocks,
        y=stocks,
        colorscale=[[0, "red"], [0.5, "hotpink"], [1, "purple"]],
        zmin=-1,
        zmax=1,
        text=round.(corr_julia, digits=3),
        texttemplate="%{text}"
    )
]
p.layout.title = "Correlation Matrix of 5 stocks"
p.layout.yaxis.autorange = "reversed"
p    

PlotlyLight.jl Plot

In [50]:
p = PlotlyLight.Plot()

p.data = [
    PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.AAPL),
        y=returns.AAPL,
        mode="lines",
        name="Apple",
        line=PlotlyLight.Config(color="hotpink")
    ),
    PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.TSLA),
        y=returns.TSLA,
        mode="lines",
        name="Tesla",
        line=PlotlyLight.Config(color="royalblue")
    ),
        PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.MSFT),
        y=returns.MSFT,
        mode="lines",
        name="Microsoft",
        line=PlotlyLight.Config(color="purple")
    ),
        PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.KO),
        y=returns.KO,
        mode="lines",
        name="Coca-Cola",
        line=PlotlyLight.Config(color="Red")
    ),
        PlotlyLight.Config(
        type="scatter",
        x=1:length(returns.JNJ),
        y=returns.JNJ,
        mode="lines",
        name="Johnson & Johnson",
        line=PlotlyLight.Config(color="teal")
    )
]

p.layout.title = "Daily Log Returns of Five Stocks"
p.layout.xaxis.title = "Trading Day"
p.layout.yaxis.title = "Daily Log Return"

p

PlotlyLight.jl Plot

In [51]:
writedlm("julia_cov_5.csv", cov_julia, ',')
writedlm("julia_corr_5.csv", corr_julia, ',')

# 10 Assets

In [52]:
stocks = ["AAPL", "TSLA", "MSFT", "KO", "JNJ", "NFLX", "V", "BA", "DIS", "CVX"]

stock_data = [
    YFinance.get_prices(
        ticker,
        startdt="2015-01-01",
        enddt="2026-01-01",
        interval="1d"
    )
    for ticker in stocks
] 

10-element Vector{OrderedCollections.OrderedDict{String, Union{Vector{Float64}, Vector{Dates.DateTime}, String}}}:
 OrderedCollections.OrderedDict("ticker" => "AAPL", "timestamp" => [Dates.DateTime("2015-01-02T14:30:00"), Dates.DateTime("2015-01-05T14:30:00"), Dates.DateTime("2015-01-06T14:30:00"), Dates.DateTime("2015-01-07T14:30:00"), Dates.DateTime("2015-01-08T14:30:00"), Dates.DateTime("2015-01-09T14:30:00"), Dates.DateTime("2015-01-12T14:30:00"), Dates.DateTime("2015-01-13T14:30:00"), Dates.DateTime("2015-01-14T14:30:00"), Dates.DateTime("2015-01-15T14:30:00")  …  Dates.DateTime("2025-12-17T14:30:00"), Dates.DateTime("2025-12-18T14:30:00"), Dates.DateTime("2025-12-19T14:30:00"), Dates.DateTime("2025-12-22T14:30:00"), Dates.DateTime("2025-12-23T14:30:00"), Dates.DateTime("2025-12-24T14:30:00"), Dates.DateTime("2025-12-26T14:30:00"), Dates.DateTime("2025-12-29T14:30:00"), Dates.DateTime("2025-12-30T14:30:00"), Dates.DateTime("2025-12-31T14:30:00")], "open" => [24.627209126024283, 23

In [53]:
prices = DataFrame(
    Date = stock_data[1]["timestamp"],
    AAPL = stock_data[1]["adjclose"],
    TSLA = stock_data[2]["adjclose"],
    MSFT = stock_data[3]["adjclose"],
    KO = stock_data[4]["adjclose"],
    JNJ = stock_data[5]["adjclose"],
    NFLX = stock_data[6]["adjclose"],
    V = stock_data[7]["adjclose"],
    BA = stock_data[8]["adjclose"],
    DIS = stock_data[9]["adjclose"],
    CVX = stock_data[10]["adjclose"]
)

Row,Date,AAPL,TSLA,MSFT,KO,JNJ,NFLX,V,BA,DIS,CVX
,DateTime,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207,39.6072,29.2159,75.7374,4.98486,60.9748,113.657,84.7168,68.3022
2,2015-01-05T14:30:00,23.4908,14.006,39.243,29.2159,75.2084,4.73114,59.6289,112.87,83.4788,65.5721
3,2015-01-06T14:30:00,23.493,14.0853,38.667,29.4378,74.8388,4.65014,59.2447,111.541,83.036,65.5418
4,2015-01-07T14:30:00,23.8224,14.0633,39.1582,29.8052,76.491,4.67429,60.0384,113.272,83.8855,65.4872
5,2015-01-08T14:30:00,24.7377,14.0413,40.3102,30.1658,77.0924,4.778,60.8437,115.275,84.7529,66.9857
6,2015-01-09T14:30:00,24.7643,13.7773,39.9714,29.833,76.0417,4.70414,59.9418,115.048,85.1686,65.651
7,2015-01-12T14:30:00,24.1541,13.4807,39.4717,29.5626,75.7809,4.55471,59.8244,114.462,85.3584,64.2374
8,2015-01-13T14:30:00,24.3685,13.6167,39.2684,29.5556,75.9113,4.62557,59.9993,114.724,86.0181,63.2181
9,2015-01-14T14:30:00,24.2757,12.846,38.9296,29.5071,75.3606,4.632,58.7983,114.033,85.1506,63.0361


In [54]:
first(prices, 5) # displays first 5 rows

Row,Date,AAPL,TSLA,MSFT,KO,JNJ,NFLX,V,BA,DIS,CVX
,DateTime,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207,39.6072,29.2159,75.7374,4.98486,60.9748,113.657,84.7168,68.3022
2,2015-01-05T14:30:00,23.4908,14.006,39.243,29.2159,75.2084,4.73114,59.6289,112.87,83.4788,65.5721
3,2015-01-06T14:30:00,23.493,14.0853,38.667,29.4378,74.8388,4.65014,59.2447,111.541,83.036,65.5418
4,2015-01-07T14:30:00,23.8224,14.0633,39.1582,29.8052,76.491,4.67429,60.0384,113.272,83.8855,65.4872
5,2015-01-08T14:30:00,24.7377,14.0413,40.3102,30.1658,77.0924,4.778,60.8437,115.275,84.7529,66.9857


In [55]:
println(size(prices)) # number rows and columns

(2766, 11)


In [56]:
[sum(ismissing, col) for col in eachcol(prices)] # check for missing values

11-element Vector{Int64}:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0

## Logarithmic Daily Returns

In [57]:
price_matrix = Matrix(prices[:, [:AAPL, :TSLA, :MSFT, :KO, :JNJ, :NFLX, :V, :BA, :DIS, :CVX]])

returns_matrix = log.(price_matrix[2:end, :] ./ price_matrix[1:end-1, :])

returns = DataFrame(returns_matrix, [:AAPL, :TSLA, :MSFT, :KO, :JNJ, :NFLX, :V, :BA, :DIS, :CVX])

Row,AAPL,TSLA,MSFT,KO,JNJ,NFLX,V,BA,DIS,CVX
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,-0.0285765,-0.0429503,-0.00923844,0.0,-0.00700906,-0.0522379,-0.022321,-0.00694986,-0.0147214,-0.0407923
2,9.43448e-5,0.00564826,-0.0147861,0.00756463,-0.00492583,-0.0172688,-0.00646431,-0.0118483,-0.00531803,-0.000462952
3,0.0139245,-0.00156315,0.0126251,0.0124055,0.021836,0.0051784,0.013309,0.0154064,0.0101777,-0.000833226
4,0.0377027,-0.00156553,0.0289942,0.0120231,0.00783175,0.0219456,0.0133234,0.0175276,0.0102882,0.0226253
5,0.00107177,-0.0189806,-0.00844077,-0.011093,-0.0137225,-0.0155784,-0.0149343,-0.00197477,0.00489235,-0.0201272
6,-0.0249492,-0.0217681,-0.0125817,-0.00910496,-0.00343654,-0.0322808,-0.00195936,-0.00510654,0.00222582,-0.0217674
7,0.00883959,0.0100379,-0.00516376,-0.000234748,0.0017197,0.015437,0.00291853,0.00228969,0.00769851,-0.0159945
8,-0.00381767,-0.0582621,-0.0086654,-0.0016431,-0.00728133,0.00138896,-0.0202196,-0.00604077,-0.0101362,-0.0028827
9,-0.0275154,-0.00426461,-0.0104988,-0.00423832,-0.0146253,-0.00148143,-0.00955416,-0.00184245,0.00127248,-0.0119092


## Sample Covariance Matrix

In [58]:
mean_returns = mean(Matrix(returns), dims=1)

1×10 Matrix{Float64}:
 0.00087428  0.00123913  0.0009027  0.000308592  …  0.000103881  0.000280322

In [59]:
centered_returns = Matrix(returns) .- mean_returns

2765×10 Matrix{Float64}:
 -0.0294507    -0.0441895   -0.0101411    …  -0.0148253   -0.0410726
 -0.000779935   0.00440913  -0.0156888       -0.00542191  -0.000743274
  0.0130502    -0.00280228   0.0117224        0.0100739   -0.00111355
  0.0368284    -0.00280466   0.0280915        0.0101843    0.022345
  0.000197491  -0.0202198   -0.00934347       0.00478847  -0.0204075
 -0.0258235    -0.0230073   -0.0134843    …   0.00212194  -0.0220478
  0.00796531    0.00879882  -0.00606646       0.00759462  -0.0162748
 -0.00469195   -0.0595012   -0.0095681       -0.0102401   -0.00316302
 -0.0283897    -0.00550374  -0.0114015        0.0011686   -0.0121895
 -0.00867476    0.00499562   0.0156696        0.00865482   0.0233027
  0.0245567    -0.00716124   0.00233642   …  -0.00473753   0.0124799
  0.00673068    0.0226488   -0.0110859       -0.0061384    0.0154679
  0.0248091     0.024127     0.0251066        0.010249     0.00672183
  ⋮                                       ⋱               
 -0.0159729    

In [60]:
n = size(returns, 1)
cov_manual = centered_returns' * centered_returns / (n - 1)
print(cov_manual)

[0.0003295799097009328 0.00028647734378403985 0.00020422545694474763 7.013562360273753e-5 6.598772247890762e-5 0.0002028690698718684 0.0001552285194438323 0.00018738952939952781 0.0001364958329401984 0.00011415900467518566; 0.00028647734378403985 0.0013158307114027078 0.00025201293971130896 5.68332518347975e-5 3.967559122591683e-5 0.0003267198407993632 0.00019232210478412079 0.0002903822877748371 0.00019404454165194017 0.00014879966700035208; 0.00020422545694474763 0.00025201293971130896 0.0002863342832056628 6.88132666673858e-5 6.486781071104406e-5 0.0002114428896639842 0.0001582614078859055 0.00016487110391312518 0.0001316443048706734 0.00010830794136602265; 7.013562360273753e-5 5.68332518347975e-5 6.88132666673858e-5 0.0001261195078831794 6.481021656642554e-5 4.257838490174603e-5 8.118095026968266e-5 0.0001049595067384972 7.343943259725329e-5 8.002274623498146e-5; 6.598772247890762e-5 3.967559122591683e-5 6.486781071104406e-5 6.481021656642554e-5 0.00013181602799702596 4.18819176281

In [61]:
cov_julia = cov(Matrix(returns))
print(cov_julia)

[0.00032957990970093273 0.00028647734378403985 0.0002042254569447476 7.013562360273753e-5 6.598772247890761e-5 0.0002028690698718684 0.00015522851944383228 0.0001873895293995278 0.0001364958329401984 0.00011415900467518566; 0.00028647734378403985 0.0013158307114027078 0.00025201293971130896 5.68332518347975e-5 3.967559122591683e-5 0.00032671984079936313 0.00019232210478412079 0.0002903822877748371 0.00019404454165194014 0.00014879966700035205; 0.0002042254569447476 0.00025201293971130896 0.0002863342832056628 6.88132666673858e-5 6.486781071104406e-5 0.0002114428896639842 0.0001582614078859055 0.00016487110391312518 0.00013164430487067337 0.00010830794136602265; 7.013562360273753e-5 5.68332518347975e-5 6.88132666673858e-5 0.00012611950788317936 6.481021656642554e-5 4.2578384901746026e-5 8.118095026968265e-5 0.0001049595067384972 7.343943259725329e-5 8.002274623498144e-5; 6.598772247890761e-5 3.967559122591683e-5 6.486781071104406e-5 6.481021656642554e-5 0.00013181602799702593 4.18819176

In [62]:
println(isapprox(cov_manual, cov_julia))

true


## Sample Correlation Matrix

In [63]:
std_returns = std(Matrix(returns), dims=1) # standard deviation

1×10 Matrix{Float64}:
 0.0181543  0.0362744  0.0169214  …  0.0252787  0.0176229  0.018468

In [64]:
corr_manual = cov_manual ./ (std_returns' * std_returns)
println(corr_manual)

[1.0000000000000002 0.43502068382993797 0.6648027944830603 0.344006908124361 0.3165910721120332 0.4093971821766817 0.5599022100606067 0.40832927530895136 0.42663985951882577 0.3404948580604316; 0.43502068382993797 0.9999999999999999 0.41056900123305223 0.13951193515483404 0.0952662976210821 0.3299779163316774 0.34717642417291317 0.3166763829484552 0.3035455156643988 0.22211746048912331; 0.6648027944830603 0.41056900123305223 1.0 0.3621132083243972 0.33389387730134734 0.45778945729587167 0.612434124962382 0.38543702761560383 0.44145638286969724 0.3465807009350381; 0.344006908124361 0.13951193515483404 0.3621132083243972 1.0000000000000002 0.5026529416171577 0.13890165687322226 0.4733517953486341 0.36972278001712316 0.37107404538178745 0.38583628486963595; 0.3165910721120332 0.0952662976210821 0.33389387730134734 0.5026529416171577 0.9999999999999998 0.13364472576318556 0.412347619817146 0.25542861260565775 0.2764817175038924 0.33653147756354496; 0.4093971821766817 0.3299779163316774 0.4

In [65]:
corr_julia = cor(Matrix(returns))
println(corr_julia)

[1.0 0.43502068382993797 0.6648027944830603 0.344006908124361 0.3165910721120332 0.40939718217668175 0.5599022100606066 0.4083292753089513 0.4266398595188257 0.3404948580604316; 0.43502068382993797 1.0 0.41056900123305234 0.13951193515483404 0.09526629762108213 0.3299779163316774 0.34717642417291317 0.31667638294845524 0.30354551566439875 0.22211746048912331; 0.6648027944830603 0.41056900123305234 1.0 0.3621132083243972 0.3338938773013474 0.4577894572958717 0.612434124962382 0.3854370276156039 0.4414563828696972 0.3465807009350382; 0.344006908124361 0.13951193515483404 0.3621132083243972 1.0 0.5026529416171578 0.13890165687322226 0.473351795348634 0.3697227800171233 0.37107404538178745 0.38583628486963595; 0.3165910721120332 0.09526629762108213 0.3338938773013474 0.5026529416171578 1.0 0.1336447257631856 0.41234761981714607 0.25542861260565786 0.2764817175038925 0.336531477563545; 0.40939718217668175 0.3299779163316774 0.4577894572958717 0.13890165687322226 0.1336447257631856 1.0 0.352

In [66]:
println(isapprox(corr_manual, corr_julia))

true


## Verifying Mathematical Properties

In [67]:
println(isapprox(cov_julia, cov_julia'))
println(isapprox(corr_julia, corr_julia'))

true
true


In [68]:
println(eigvals(cov_julia))
println(eigvals(corr_julia))

[6.20007236632792e-5, 8.46213224054797e-5, 0.00010302489012733563, 0.00012867953568623756, 0.0001768607410561945, 0.00021161608360879778, 0.0003028861499237282, 0.0005780773751181688, 0.0007541903702638614, 0.002056626650493309]
[0.3159300775822249, 0.3885641358237017, 0.46275369564893926, 0.5101595353519575, 0.5503465064439668, 0.563945067633031, 0.667687108488476, 0.8684790644089407, 1.2675449900008733, 4.404589818617888]


In [69]:
println(sum(eigvals(corr_julia)))

10.0


Sum of the eigenvalues of the correlation matrix is equal to the number of assets used.

In [70]:
println(minimum(eigvals(Symmetric(cov_julia))) >= -1e-10)
println(minimum(eigvals(Symmetric(corr_julia))) >= -1e-10)

true
true


In [71]:
eigenvectors = eigen(Symmetric(corr_julia)).vectors
println(eigenvectors)

[0.5144759926521408 0.5189201827284643 0.10831725575693012 -0.18103164519027337 0.16813339427414783 0.3809682757761161 0.1765736397394267 -0.19182799409389303 0.22161386073713038 -0.36198979851501023; -0.02515203760805513 -0.06674763075552068 -0.07217183604958319 0.20819577246282822 -0.21517540765905607 -0.38339804979026154 0.6797902018459523 0.1780921003770889 0.441953150160318 -0.2528027944449912; -0.7608595601176714 -0.04152202749462676 -0.03866278939324966 -0.06891967672819524 0.16823270482389244 0.3642518433962294 0.03455176503701325 -0.2601684892270833 0.21413847085744148 -0.3706825023403989; -0.018561402286403468 0.15947392090956874 -0.6075775305026792 -0.3208943975037225 -0.3737395538099224 -0.1648158202921925 0.07387579966438074 -0.20861475626133755 -0.4473931551683667 -0.293455241349879; -0.02745628868157144 0.006190490210408975 0.5499016254760308 0.23637977570926869 0.09542097124913687 -0.34196586746200985 0.14564347114686 -0.4696570860834846 -0.4545968019205899 -0.257772702

In [72]:
eigenvectors_df = DataFrame(eigenvectors, :auto)

rename!(eigenvectors_df, [
    "Eigenvector 1", "Eigenvector 2",
    "Eigenvector 3", "Eigenvector 4", "Eigenvector 5",
    "Eigenvector 6", "Eigenvector 7", "Eigenvector 8",
    "Eigenvector 9", "Eigenvector 10"
])

eigenvectors_df.Stock = ["AAPL", "TSLA", "MSFT", "KO", "JNJ", "NFLX", "V", "BA", "DIS", "CVX"]

select!(eigenvectors_df, :Stock, :)

eigenvectors_df

Row,Stock,Eigenvector 1,Eigenvector 2,Eigenvector 3,Eigenvector 4,Eigenvector 5,Eigenvector 6,Eigenvector 7,Eigenvector 8,Eigenvector 9,Eigenvector 10
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,AAPL,0.514476,0.51892,0.108317,-0.181032,0.168133,0.380968,0.176574,-0.191828,0.221614,-0.36199
2,TSLA,-0.025152,-0.0667476,-0.0721718,0.208196,-0.215175,-0.383398,0.67979,0.178092,0.441953,-0.252803
3,MSFT,-0.76086,-0.041522,-0.0386628,-0.0689197,0.168233,0.364252,0.0345518,-0.260168,0.214138,-0.370683
4,KO,-0.0185614,0.159474,-0.607578,-0.320894,-0.37374,-0.164816,0.0738758,-0.208615,-0.447393,-0.293455
5,JNJ,-0.0274563,0.00619049,0.549902,0.23638,0.095421,-0.341966,0.145643,-0.469657,-0.454597,-0.257773
6,NFLX,0.0957304,0.00201307,-0.129891,-0.0295237,0.134075,-0.528323,-0.582583,-0.250729,0.460515,-0.248284
7,V,0.366727,-0.781701,-0.0811802,0.0778789,-0.0309285,0.289245,-0.0411422,-0.0620334,-0.044289,-0.386881
8,BA,-0.0823932,-0.0829119,0.400631,-0.65055,0.00689067,-0.171806,-0.062075,0.509596,-0.083979,-0.32009
9,DIS,-0.0642198,0.249491,0.174008,0.454208,-0.582502,0.178612,-0.365481,0.283301,-0.0231533,-0.333862


In [73]:
println(all(corr_julia .>= -1))
println(all(corr_julia .<= 1))

true
true


## Visuals

In [74]:
p = PlotlyLight.Plot()
p.data = [
    PlotlyLight.Config(
        type="heatmap",
        z=corr_julia,
        x=stocks,
        y=stocks,
        colorscale=[[0, "red"], [0.5, "hotpink"], [1, "purple"]],
        zmin=-1,
        zmax=1,
        text=round.(corr_julia, digits=3),
        texttemplate="%{text}"
    )
]
p.layout.title = "Correlation Matrix of 10 stocks"
p.layout.yaxis.autorange = "reversed"
p   

PlotlyLight.jl Plot

In [75]:
writedlm("julia_cov_10.csv", cov_julia, ',')
writedlm("julia_corr_10.csv", corr_julia, ',')

# 50 Assets

In [76]:
stocks = [
    "AAPL", "TSLA", "MSFT", "KO", "JNJ", "NFLX", "V", "BA", "DIS", "CVX",
    "NVDA", "AMD", "INTC", "CSCO", "ORCL", "IBM", "CRM", "GOOGL", "META", "CMCSA",
    "AMZN", "HD", "MCD", "NKE", "SBUX","PEP", "PG", "WMT", "COST", "CL","PFE", "MRK", "ABBV", "ABT", "TMO",
    "JPM", "BAC", "WFC", "GS", "MA","XOM", "COP", "SLB","CAT", "DE", "UNP", "UPS","NEE", "DUK","LIN"
]
stock_data = [
    YFinance.get_prices(
        ticker,
        startdt="2015-01-01",
        enddt="2026-01-01",
        interval="1d"
    )
    for ticker in stocks
]

50-element Vector{OrderedCollections.OrderedDict{String, Union{Vector{Float64}, Vector{Dates.DateTime}, String}}}:
 OrderedCollections.OrderedDict("ticker" => "AAPL", "timestamp" => [Dates.DateTime("2015-01-02T14:30:00"), Dates.DateTime("2015-01-05T14:30:00"), Dates.DateTime("2015-01-06T14:30:00"), Dates.DateTime("2015-01-07T14:30:00"), Dates.DateTime("2015-01-08T14:30:00"), Dates.DateTime("2015-01-09T14:30:00"), Dates.DateTime("2015-01-12T14:30:00"), Dates.DateTime("2015-01-13T14:30:00"), Dates.DateTime("2015-01-14T14:30:00"), Dates.DateTime("2015-01-15T14:30:00")  …  Dates.DateTime("2025-12-17T14:30:00"), Dates.DateTime("2025-12-18T14:30:00"), Dates.DateTime("2025-12-19T14:30:00"), Dates.DateTime("2025-12-22T14:30:00"), Dates.DateTime("2025-12-23T14:30:00"), Dates.DateTime("2025-12-24T14:30:00"), Dates.DateTime("2025-12-26T14:30:00"), Dates.DateTime("2025-12-29T14:30:00"), Dates.DateTime("2025-12-30T14:30:00"), Dates.DateTime("2025-12-31T14:30:00")], "open" => [24.62720329616351, 23.

In [77]:
prices = DataFrame(
    Date = stock_data[1]["timestamp"], 
    AAPL = stock_data[1]["adjclose"],
    TSLA = stock_data[2]["adjclose"],
    MSFT = stock_data[3]["adjclose"],
    KO = stock_data[4]["adjclose"],
    JNJ = stock_data[5]["adjclose"],
    NFLX = stock_data[6]["adjclose"],
    V = stock_data[7]["adjclose"],
    BA = stock_data[8]["adjclose"],
    DIS = stock_data[9]["adjclose"],
    CVX = stock_data[10]["adjclose"],
    NVDA = stock_data[11]["adjclose"],
    AMD = stock_data[12]["adjclose"], 
    INTC = stock_data[13]["adjclose"], 
    CSCO = stock_data[14]["adjclose"],
    ORCL = stock_data[15]["adjclose"],
    IBM = stock_data[16]["adjclose"],
    CRM = stock_data[17]["adjclose"],
    GOOGL = stock_data[18]["adjclose"],
    META = stock_data[19]["adjclose"],
    CMCSA = stock_data[20]["adjclose"],
    AMZN = stock_data[21]["adjclose"],
    HD = stock_data[22]["adjclose"], 
    MCD = stock_data[23]["adjclose"], 
    NKE = stock_data[24]["adjclose"],
    SBUX = stock_data[25]["adjclose"],
    PEP = stock_data[26]["adjclose"],
    PG = stock_data[27]["adjclose"], 
    WMT = stock_data[28]["adjclose"],
    COST = stock_data[29]["adjclose"],
    CL = stock_data[30]["adjclose"],
    PFE = stock_data[31]["adjclose"],
    MRK = stock_data[32]["adjclose"], 
    ABBV = stock_data[33]["adjclose"],
    ABT = stock_data[34]["adjclose"],
    TMO = stock_data[35]["adjclose"],
    JPM = stock_data[36]["adjclose"],
    BAC = stock_data[37]["adjclose"],
    WFC = stock_data[38]["adjclose"],
    GS = stock_data[39]["adjclose"],
    MA = stock_data[40]["adjclose"],
    XOM = stock_data[41]["adjclose"],
    COP = stock_data[42]["adjclose"],
    SLB = stock_data[43]["adjclose"],
    CAT = stock_data[44]["adjclose"],
    DE = stock_data[45]["adjclose"],
    UNP = stock_data[46]["adjclose"],
    UPS = stock_data[47]["adjclose"],
    NEE = stock_data[48]["adjclose"],
    DUK = stock_data[49]["adjclose"],
    LIN = stock_data[50]["adjclose"],
    
)

Row,Date,AAPL,TSLA,MSFT,KO,JNJ,NFLX,V,BA,DIS,CVX,NVDA,AMD,INTC,CSCO,ORCL,IBM,CRM,GOOGL,META,CMCSA,AMZN,HD,MCD,NKE,SBUX,PEP,PG,WMT,COST,CL,PFE,MRK,ABBV,ABT,TMO,JPM,BAC,WFC,GS,MA,XOM,COP,SLB,CAT,DE,UNP,UPS,NEE,DUK,LIN
,DateTime,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207,39.6072,29.2159,75.7374,4.98486,60.9748,113.657,84.7168,68.3023,0.481884,2.67,27.8965,19.4554,37.0869,96.1641,58.1303,26.2279,77.7058,19.7982,15.426,78.2041,69.3118,40.3164,32.0767,65.8792,64.6421,23.0723,113.467,52.1456,17.5159,37.9853,40.8683,35.5354,123.58,45.8381,13.9795,39.2763,154.118,79.594,56.7779,47.2486,61.8874,68.7358,72.0924,91.2808,71.034,19.6051,52.0014,105.288
2,2015-01-05T14:30:00,23.4908,14.006,39.243,29.2159,75.2084,4.73114,59.6289,112.87,83.4788,65.5721,0.473745,2.66,27.5819,19.0679,36.5668,94.651,57.0803,25.7282,76.4577,19.3945,15.1095,76.5633,68.5463,39.6673,31.4623,65.3839,64.3347,23.0052,112.177,51.7605,17.4208,38.5499,40.0992,35.5434,121.882,44.4151,13.5734,38.1993,149.306,77.3552,55.2244,45.0,60.2114,65.1076,70.55,88.1947,69.6118,19.3974,51.2527,102.509
3,2015-01-06T14:30:00,23.493,14.0853,38.667,29.4378,74.8388,4.65014,59.2447,111.541,83.036,65.5418,0.459381,2.63,27.0678,19.0608,36.1893,92.6097,56.1285,25.0932,75.4276,19.1415,14.7645,76.3289,68.6726,39.434,31.2063,64.8886,64.0417,23.1824,113.659,51.2166,17.5662,40.0643,39.9007,35.1397,120.744,43.2634,13.1673,37.4023,146.285,77.188,54.9308,43.1421,59.0339,64.6886,69.9625,86.3708,69.1549,19.4728,52.0941,101.197
4,2015-01-07T14:30:00,23.8224,14.0633,39.1583,29.8052,76.491,4.67429,60.0384,113.272,83.8855,65.4872,0.458184,2.58,27.6356,19.237,36.1977,92.0045,55.8636,25.0194,75.4276,19.1241,14.921,78.945,69.8692,40.2485,31.9743,66.786,64.3776,23.7975,115.638,51.6245,17.8066,40.9211,41.5134,35.4246,124.361,43.3294,13.2297,37.6249,148.466,78.3887,55.4874,43.4301,59.0267,65.6911,69.6605,86.848,69.7984,19.651,52.2612,102.331
5,2015-01-08T14:30:00,24.7377,14.0413,40.3102,30.1657,77.0924,4.778,60.8437,115.275,84.7529,66.9857,0.47542,2.61,28.1496,19.385,36.4158,94.0042,57.4925,25.1066,77.4383,19.54,15.023,80.6916,70.1293,41.1776,32.4903,67.9998,65.1138,24.2998,116.632,52.3269,18.17,41.7447,41.9475,36.1528,125.654,44.2977,13.5031,38.4578,150.836,79.608,56.4109,44.5133,59.7418,66.3644,70.5827,90.1033,71.0533,19.7705,52.4715,104.016
6,2015-01-09T14:30:00,24.7643,13.7773,39.9714,29.833,76.0417,4.70414,59.9418,115.048,85.1686,65.651,0.477335,2.63,28.2033,19.5823,36.399,94.4136,57.0705,24.8,77.0025,19.5088,14.8465,79.308,69.2747,40.7237,31.4268,67.5394,64.5063,23.999,114.837,51.8133,18.2539,41.5521,40.8001,35.7729,125.152,43.5275,13.261,37.8259,148.521,78.5934,56.3314,44.5064,58.6727,65.5714,69.8972,88.4872,70.5321,19.5959,52.5458,102.614
7,2015-01-12T14:30:00,24.1541,13.4807,39.4716,29.5626,75.7808,4.55471,59.8244,114.462,85.3584,64.2374,0.471351,2.63,28.0806,19.7655,36.2983,92.8293,56.6486,24.6187,75.9922,19.4673,14.5705,78.9526,69.126,40.6685,31.6001,67.2604,64.2704,24.1789,114.196,51.5565,18.321,41.3794,40.7877,36.0736,123.946,43.1534,13.0267,37.3736,146.714,77.5882,55.2488,43.2381,56.3827,65.0028,69.734,87.7099,70.5578,19.5463,52.5953,102.388
8,2015-01-13T14:30:00,24.3685,13.6167,39.2684,29.5556,75.9113,4.62557,59.9993,114.724,86.0181,63.2181,0.470633,2.66,28.0039,19.8007,36.0131,93.0489,56.8841,24.8535,75.7247,19.6995,14.737,78.4309,68.9922,40.3079,31.8522,67.2115,64.542,23.9882,113.05,51.9115,18.1365,41.3063,39.6128,35.4928,123.792,43.1607,12.8471,37.23,146.603,77.7651,55.047,42.8062,56.1877,64.6138,70.093,86.063,70.8603,19.6234,52.8799,101.683
9,2015-01-14T14:30:00,24.2757,12.846,38.9296,29.5071,75.3605,4.632,58

In [78]:
first(prices, 5) # displays first 5 rows

Row,Date,AAPL,TSLA,MSFT,KO,JNJ,NFLX,V,BA,DIS,CVX,NVDA,AMD,INTC,CSCO,ORCL,IBM,CRM,GOOGL,META,CMCSA,AMZN,HD,MCD,NKE,SBUX,PEP,PG,WMT,COST,CL,PFE,MRK,ABBV,ABT,TMO,JPM,BAC,WFC,GS,MA,XOM,COP,SLB,CAT,DE,UNP,UPS,NEE,DUK,LIN
,DateTime,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2015-01-02T14:30:00,24.1718,14.6207,39.6072,29.2159,75.7374,4.98486,60.9748,113.657,84.7168,68.3023,0.481884,2.67,27.8965,19.4554,37.0869,96.1641,58.1303,26.2279,77.7058,19.7982,15.426,78.2041,69.3118,40.3164,32.0767,65.8792,64.6421,23.0723,113.467,52.1456,17.5159,37.9853,40.8683,35.5354,123.58,45.8381,13.9795,39.2763,154.118,79.594,56.7779,47.2486,61.8874,68.7358,72.0924,91.2808,71.034,19.6051,52.0014,105.288
2,2015-01-05T14:30:00,23.4908,14.006,39.243,29.2159,75.2084,4.73114,59.6289,112.87,83.4788,65.5721,0.473745,2.66,27.5819,19.0679,36.5668,94.651,57.0803,25.7282,76.4577,19.3945,15.1095,76.5633,68.5463,39.6673,31.4623,65.3839,64.3347,23.0052,112.177,51.7605,17.4208,38.5499,40.0992,35.5434,121.882,44.4151,13.5734,38.1993,149.306,77.3552,55.2244,45.0,60.2114,65.1076,70.55,88.1947,69.6118,19.3974,51.2527,102.509
3,2015-01-06T14:30:00,23.493,14.0853,38.667,29.4378,74.8388,4.65014,59.2447,111.541,83.036,65.5418,0.459381,2.63,27.0678,19.0608,36.1893,92.6097,56.1285,25.0932,75.4276,19.1415,14.7645,76.3289,68.6726,39.434,31.2063,64.8886,64.0417,23.1824,113.659,51.2166,17.5662,40.0643,39.9007,35.1397,120.744,43.2634,13.1673,37.4023,146.285,77.188,54.9308,43.1421,59.0339,64.6886,69.9625,86.3708,69.1549,19.4728,52.0941,101.197
4,2015-01-07T14:30:00,23.8224,14.0633,39.1583,29.8052,76.491,4.67429,60.0384,113.272,83.8855,65.4872,0.458184,2.58,27.6356,19.237,36.1977,92.0045,55.8636,25.0194,75.4276,19.1241,14.921,78.945,69.8692,40.2485,31.9743,66.786,64.3776,23.7975,115.638,51.6245,17.8066,40.9211,41.5134,35.4246,124.361,43.3294,13.2297,37.6249,148.466,78.3887,55.4874,43.4301,59.0267,65.6911,69.6605,86.848,69.7984,19.651,52.2612,102.331
5,2015-01-08T14:30:00,24.7377,14.0413,40.3102,30.1657,77.0924,4.778,60.8437,115.275,84.7529,66.9857,0.47542,2.61,28.1496,19.385,36.4158,94.0042,57.4925,25.1066,77.4383,19.54,15.023,80.6916,70.1293,41.1776,32.4903,67.9998,65.1138,24.2998,116.632,52.3269,18.17,41.7447,41.9475,36.1528,125.654,44.2977,13.5031,38.4578,150.836,79.608,56.4109,44.5133,59.7418,66.3644,70.5827,90.1033,71.0533,19.7705,52.4715,104.016


In [79]:
println(size(prices)) # number rows and columns

(2766, 51)


In [80]:
[sum(ismissing, col) for col in eachcol(prices)] # check for missing values

51-element Vector{Int64}:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0

In [81]:
println(all(
    stock_data[i]["timestamp"] == stock_data[1]["timestamp"]
    for i in 2:length(stocks)
))

true


## Logarithmic Daily Returns

In [82]:
price_matrix = Matrix(prices[:, [:AAPL, :TSLA, :MSFT, :KO, :JNJ, :NFLX, :V, :BA, :DIS, :CVX, :NVDA, :AMD, :INTC,
            :CSCO, :ORCL, :IBM, :CRM, :GOOGL, :META, :CMCSA, :AMZN, :HD, :MCD, :NKE, :SBUX, :PEP, :PG, :WMT, :COST, 
            :CL,:PFE, :MRK, :ABBV, :ABT, :TMO, :JPM, :BAC, :WFC, :GS, :MA, :XOM, :COP, :SLB,:CAT, :DE, :UNP, :UPS,
            :NEE, :DUK,:LIN]])

returns_matrix = log.(price_matrix[2:end, :] ./ price_matrix[1:end-1, :])

returns = DataFrame(returns_matrix, [:AAPL, :TSLA, :MSFT, :KO, :JNJ, :NFLX, :V, :BA, :DIS, :CVX, :NVDA, :AMD, :INTC,
            :CSCO, :ORCL, :IBM, :CRM, :GOOGL, :META, :CMCSA, :AMZN, :HD, :MCD, :NKE, :SBUX, :PEP, :PG, :WMT, :COST, 
            :CL,:PFE, :MRK, :ABBV, :ABT, :TMO, :JPM, :BAC, :WFC, :GS, :MA, :XOM, :COP, :SLB,:CAT, :DE, :UNP, :UPS,
            :NEE, :DUK,:LIN])

Row,AAPL,TSLA,MSFT,KO,JNJ,NFLX,V,BA,DIS,CVX,NVDA,AMD,INTC,CSCO,ORCL,IBM,CRM,GOOGL,META,CMCSA,AMZN,HD,MCD,NKE,SBUX,PEP,PG,WMT,COST,CL,PFE,MRK,ABBV,ABT,TMO,JPM,BAC,WFC,GS,MA,XOM,COP,SLB,CAT,DE,UNP,UPS,NEE,DUK,LIN
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,-0.028576,-0.0429503,-0.00923834,0.0,-0.00700926,-0.0522379,-0.0223212,-0.00694972,-0.0147211,-0.0407924,-0.0170344,-0.00375235,-0.0113402,-0.0201216,-0.0141235,-0.01586,-0.0182273,-0.0192377,-0.0161916,-0.0206045,-0.0207307,-0.0212034,-0.0111059,-0.0162308,-0.0193416,-0.00754684,-0.00476599,-0.00291442,-0.0114346,-0.00741414,-0.00544078,0.0147534,-0.0189984,0.000222832,-0.0138322,-0.0315374,-0.0294808,-0.0278053,-0.0317205,-0.0285309,-0.0277431,-0.0487616,-0.0274541,-0.0542301,-0.0216267,-0.0343935,-0.0202241,-0.0106478,-0.0145013,-0.0267498
2,9.42636e-5,0.00564826,-0.0147863,0.00756515,-0.00492573,-0.0172688,-0.00646457,-0.0118485,-0.0053183,-0.000462836,-0.0307876,-0.0113423,-0.0188131,-0.000369777,-0.0103772,-0.0218022,-0.0168159,-0.0249891,-0.0135649,-0.0131313,-0.0230981,-0.00306639,0.0018417,-0.00589964,-0.00817058,-0.00760387,-0.00456494,0.00767633,0.0131278,-0.010562,0.00830914,0.0385313,-0.00496198,-0.0114212,-0.00938207,-0.0262709,-0.0303763,-0.0210855,-0.0204371,-0.00216427,-0.00533064,-0.0421621,-0.0197495,-0.00645534,-0.00836351,-0.0208971,-0.00658575,0.00387641,0.0162841,-0.0128868
3,0.0139246,-0.00156315,0.0126257,0.0124052,0.0218356,0.0051784,0.0133089,0.0154064,0.0101778,-0.000833225,-0.0026091,-0.0191945,0.0207583,0.00920019,0.000231663,-0.00655675,-0.00473139,-0.00294526,0.0,-0.000905783,0.010544,0.0337003,0.0172741,0.0204458,0.0243141,0.0288218,0.00523166,0.0261864,0.0172634,0.00793173,0.0135928,0.0211607,0.0396214,0.00807539,0.0295166,0.00152468,0.00473374,0.00593354,0.014793,0.0154355,0.010082,0.0066518,-0.00012246,0.0153779,-0.00432521,0.00550905,0.00926274,0.00911184,0.0032017,0.0111468
4,0.0377023,-0.00156553,0.0289939,0.0120229,0.00783214,0.0219456,0.0133237,0.0175276,0.010288,0.0226254,0.0369275,0.0115608,0.0184299,0.00766263,0.00600763,0.0215023,0.0287416,0.00347802,0.0263088,0.021514,0.00681276,0.0218828,0.0037161,0.0228217,0.016008,0.0180112,0.0113703,0.0208863,0.00855495,0.013515,0.0202029,0.0199262,0.0104042,0.020346,0.0103393,0.0221004,0.0204509,0.021896,0.0158392,0.0154351,0.0165074,0.0246347,0.0120429,0.010197,0.0131515,0.036798,0.0178192,0.00605957,0.00401675,0.0163346
5,0.00107216,-0.0189806,-0.00844087,-0.0110927,-0.0137228,-0.0155784,-0.0149343,-0.00197463,0.00489271,-0.0201275,0.00402,0.00763371,0.00190589,0.0101269,-0.000460814,0.0043456,-0.00736631,-0.0122864,-0.00564398,-0.00159753,-0.0118182,-0.017296,-0.012262,-0.0110855,-0.0332793,-0.00679346,-0.00937383,-0.0124569,-0.0155081,-0.00986383,0.0046047,-0.00462435,-0.0277361,-0.0105637,-0.00399987,-0.0175402,-0.0180919,-0.0165666,-0.0154654,-0.0128263,-0.0014106,-0.000153925,-0.018058,-0.012021,-0.0097595,-0.0180989,-0.00736348,-0.00886867,0.00141389,-0.0135674
6,-0.0249497,-0.0217681,-0.0125818,-0.00910483,-0.00343644,-0.0322808,-0.00195968,-0.00510654,0.00222564,-0.021767,-0.0126166,0.0,-0.00436193,0.00931205,-0.00276956,-0.0169231,-0.00742071,-0.00733629,-0.0132075,-0.00213392,-0.0187652,-0.00449124,-0.00214802,-0.00135486,0.00549966,-0.00414004,-0.00366349,0.00747038,-0.00559785,-0.00496876,0.00366849,-0.00416514,-0.000303632,0.00837208,-0.00968019,-0.00863162,-0.0178262,-0.0120308,-0.0122442,-0.0128722,-0.0194049,-0.0289105,-0.0398119,-0.00870875,-0.00233792,-0.00882345,0.000365437,-0.00253526,0.00094151,-0.00221347
7,0.00883967,0.0100379,-0.00516347,-0.000234748,0.0017198,0.015437,0.00291904,0.00228962,0.00769868,-0.0159944,-0.

## Sample Covariance Matrix

In [83]:
mean_returns = mean(Matrix(returns), dims=1)

1×50 Matrix{Float64}:
 0.00087428  0.00123913  0.0009027  0.000308592  …  0.000284578  0.000502298

In [84]:
centered_returns = Matrix(returns) .- mean_returns

2765×50 Matrix{Float64}:
 -0.0294503    -0.0441895   -0.010141     …  -0.0147859    -0.0272521
 -0.000780016   0.00440913  -0.015689         0.0159996    -0.0133891
  0.0130503    -0.00280228   0.011723         0.00291712    0.0106445
  0.0368281    -0.00280466   0.0280912        0.00373217    0.0158323
  0.000197876  -0.0202198   -0.00934357       0.00112932   -0.0140697
 -0.0258239    -0.0230073   -0.0134845    …   0.000656932  -0.00271577
  0.00796539    0.00879882  -0.00606617       0.00511224   -0.00741053
 -0.0046921    -0.0595012   -0.00956771       0.0022861    -0.00777998
 -0.0283896    -0.00550374  -0.011402         0.0121249    -0.00146557
 -0.0086746     0.00499562   0.01567          0.00649288   -0.0080831
  0.0245568    -0.00716124   0.00233593   …   0.00359999    0.0106073
  0.00673092    0.0226488   -0.0110856        0.0109418     0.00556351
  0.0248087     0.024127     0.0251061       -0.000848607   0.0136408
  ⋮                                       ⋱                


In [85]:
n = size(returns, 1)
cov_manual = centered_returns' * centered_returns / (n - 1)
print(cov_manual)

[0.0003295797720739375 0.00028647750531033203 0.0002042253267631594 7.013553806203697e-5 6.59876463551428e-5 0.00020286903502939323 0.00015522853643888475 0.0001873895252384111 0.00013649595069250745 0.00011415882769557312 0.00029458972946094986 0.00028394702728266717 0.0002007638736261115 0.00015085253110634457 0.00014052717840759004 0.0001103369827138935 0.00019976961541916187 0.00019516065266257402 0.00022462724043537677 0.0001163553176993639 0.00020920452519534113 0.00013864058316503784 8.949743841711483e-5 0.00015801803021836997 0.0001494940749203827 8.713868519924942e-5 7.48640398917759e-5 7.877256189154615e-5 0.00011706796085544479 6.52731902126031e-5 7.907826566589594e-5 7.221633757477777e-5 8.477913712612573e-5 0.000115196729287899 0.00013514079033643964 0.0001322582252898017 0.00014871237848647657 0.00013303111910158895 0.00015822829226301002 0.0001738759139448724 9.818502488035159e-5 0.00013385790590994312 0.00013027197739872332 0.00013482721777161952 0.00012993798260431343 

In [86]:
cov_julia = cov(Matrix(returns))
print(cov_julia)

[0.0003295797720739374 0.000286477505310332 0.0002042253267631594 7.013553806203696e-5 6.598764635514279e-5 0.00020286903502939323 0.00015522853643888473 0.0001873895252384111 0.00013649595069250745 0.0001141588276955731 0.00029458972946094986 0.00028394702728266717 0.00020076387362611146 0.00015085253110634454 0.00014052717840759004 0.0001103369827138935 0.00019976961541916184 0.00019516065266257402 0.00022462724043537677 0.00011635531769936388 0.0002092045251953411 0.00013864058316503784 8.949743841711483e-5 0.00015801803021836997 0.00014949407492038268 8.713868519924941e-5 7.486403989177588e-5 7.877256189154613e-5 0.00011706796085544479 6.52731902126031e-5 7.907826566589594e-5 7.221633757477776e-5 8.477913712612573e-5 0.00011519672928789899 0.00013514079033643961 0.0001322582252898017 0.00014871237848647657 0.00013303111910158895 0.00015822829226301002 0.00017387591394487238 9.818502488035159e-5 0.0001338579059099431 0.00013027197739872332 0.00013482721777161952 0.000129937982604313

In [87]:
println(isapprox(cov_manual, cov_julia))

true


## Sample Correlation Matrix

In [88]:
std_returns = std(Matrix(returns), dims=1) # standard deviation

1×50 Matrix{Float64}:
 0.0181543  0.0362744  0.0169214  …  0.0155828  0.012808  0.0141585

In [89]:
corr_manual = cov_manual ./ (std_returns' * std_returns)
println(corr_manual)

[1.0000000000000002 0.43502101993905923 0.6648025690781887 0.3440067032795861 0.3165906405241601 0.4093971973419772 0.5599020028750354 0.4083293477275868 0.4266403207158787 0.34049434738610895 0.5326556750931649 0.4294613619132066 0.4476276726385205 0.5277960311707681 0.38957912657171273 0.3936190285096646 0.5042954495671615 0.5929046541420367 0.5190349008709043 0.3998712937815366 0.5576592778373682 0.4960192263037835 0.3868341673206233 0.44402868162107423 0.45743553056926567 0.3997371307829177 0.35369488988166037 0.3204574340962927 0.4733543278922829 0.29673592823549405 0.2947826574781102 0.28011704677180965 0.27689036073806383 0.4285380613259536 0.46438179788835526 0.42656014686591437 0.4231545903407708 0.3724258166763575 0.4765690157401151 0.5775244843161187 0.31296025234330394 0.3071055269780803 0.2897550611336028 0.3929943945093818 0.3868209403553989 0.4365218932190011 0.4237185938667398 0.3092551258375836 0.2445054026988015 0.48129882149781095; 0.43502101993905923 0.9999999999999

In [90]:
corr_julia = cor(Matrix(returns))
println(corr_julia)

[1.0 0.4350210199390592 0.6648025690781887 0.344006703279586 0.31659064052416 0.40939719734197716 0.5599020028750353 0.4083293477275868 0.4266403207158786 0.3404943473861089 0.5326556750931649 0.4294613619132065 0.4476276726385205 0.5277960311707679 0.3895791265717127 0.3936190285096645 0.5042954495671614 0.5929046541420365 0.5190349008709042 0.39987129378153646 0.557659277837368 0.4960192263037833 0.3868341673206232 0.44402868162107395 0.4574355305692655 0.39973713078291767 0.3536948898816603 0.3204574340962927 0.4733543278922829 0.296735928235494 0.29478265747811017 0.2801170467718096 0.2768903607380638 0.4285380613259536 0.4643817978883551 0.4265601468659143 0.42315459034077063 0.37242581667635744 0.47656901574011507 0.5775244843161185 0.3129602523433039 0.30710552697808025 0.28975506113360266 0.3929943945093817 0.38682094035539877 0.4365218932190009 0.42371859386673966 0.30925512583758347 0.2445054026988015 0.48129882149781095; 0.4350210199390592 1.0 0.4105687441557045 0.1395119701

In [91]:
println(isapprox(corr_manual, corr_julia))

true


## Verifying Mathematical Properties

In [88]:
println(isapprox(cov_julia, cov_julia'))
println(isapprox(corr_julia, corr_julia'))

true
true


In [89]:
println(eigvals(cov_julia))
println(eigvals(corr_julia))

[2.5267977575460283e-5, 3.188739362395988e-5, 3.4152564588573234e-5, 3.610911541564212e-5, 4.56648053291414e-5, 4.8932586151762976e-5, 5.2694244500743945e-5, 5.6254196151226406e-5, 5.841569634428108e-5, 7.165260507525041e-5, 7.199586905687607e-5, 7.480019733712727e-5, 7.940399049533525e-5, 8.168615119647472e-5, 8.639750080066497e-5, 9.096568938641213e-5, 9.56330437587985e-5, 0.0001005904741214497, 0.00010783442570441455, 0.00010821673724814534, 0.00011486346470530566, 0.000116086659697856, 0.0001202099781713038, 0.00012714748284206023, 0.00012831353683100013, 0.00013393967792850128, 0.00014247407698039617, 0.00014321052389649257, 0.00015656970875695234, 0.00016166475486696196, 0.0001704441207242063, 0.00017953466985222696, 0.00019471837559669497, 0.00019923065869207005, 0.00020547149853841025, 0.00024294796908878345, 0.00026266848587705796, 0.00027072148693676427, 0.0002908631247785605, 0.00030080268908348897, 0.00034856656888963274, 0.00038138753799277946, 0.0004017556878222114, 0.000

In [92]:
println(sum(eigvals(corr_julia)))

50.0


In [93]:
eigenvectors = eigen(Symmetric(corr_julia)).vectors
println(eigenvectors)

[-0.019472712465419813 -0.008007139392960551 0.04459967340069095 -0.037238614718805955 -0.04994759689013309 0.03157438606664044 -0.14109486437124735 0.05777798607538435 -0.051082705474131254 -0.1471259166684775 0.006441932950225722 0.14804713167872705 0.00565022528665186 -0.03491999350436609 0.1817371577440462 -0.27112264250715823 0.07724444097228811 0.5324102188176952 -0.1443983077985962 -0.31131047894837083 0.07543785359909325 -0.25913650177487807 -0.21579975685307962 0.16606894175822656 -0.14538542024025777 0.11614085121945147 -0.06124279898915429 0.19503523549588433 -0.14660663322670597 -0.06851859139822569 0.0336456022342134 0.0756459812861535 -0.06841708254365494 -0.12175291601366811 -0.005242831215272116 0.22232864592616808 0.10680661185652515 0.0518850298007682 -0.006971166629741169 -0.03242480141859318 -0.016359560772972227 0.021615318019166076 -0.02527239431334351 -0.02909155514959754 -0.010460068697580361 -0.08004561693192702 0.010795929022575222 -0.14827261655372714 0.10768

In [93]:
eigenvectors_df = DataFrame(eigenvectors, :auto)

rename!(eigenvectors_df, [
    "Eigenvector 1", "Eigenvector 2",
    "Eigenvector 3", "Eigenvector 4", "Eigenvector 5",
    "Eigenvector 6", "Eigenvector 7", "Eigenvector 8",
    "Eigenvector 9", "Eigenvector 10", "Eigenvector 11",
    "Eigenvector 12", "Eigenvector 13", "Eigenvector 14",
    "Eigenvector 15", "Eigenvector 16", "Eigenvector 17",
    "Eigenvector 18", "Eigenvector 19", "Eigenvector 20",
    "Eigenvector 21", "Eigenvector 22", "Eigenvector 23",
    "Eigenvector 24", "Eigenvector 25", "Eigenvector 26",
    "Eigenvector 27", "Eigenvector 28", "Eigenvector 29",
    "Eigenvector 30", "Eigenvector 31", "Eigenvector 32",
    "Eigenvector 33", "Eigenvector 34", "Eigenvector 35",
    "Eigenvector 36", "Eigenvector 37", "Eigenvector 38", "Eigenvector 39",
    "Eigenvector 40", "Eigenvector 41", "Eigenvector 42",
    "Eigenvector 43", "Eigenvector 44", "Eigenvector 45",
    "Eigenvector 46", "Eigenvector 47", "Eigenvector 48",
    "Eigenvector 49", "Eigenvector 50"    
])

eigenvectors_df.Stock = ["AAPL", "TSLA", "MSFT", "KO", "JNJ", "NFLX", "V", "BA", "DIS", "CVX", 
    "NVDA", "AMD", "INTC", "CSCO", "ORCL", "IBM", "CRM", "GOOGL", "META", "CMCSA",
    "AMZN", "HD", "MCD", "NKE", "SBUX","PEP", "PG", "WMT", "COST", "CL","PFE", "MRK", "ABBV", "ABT", "TMO",
    "JPM", "BAC", "WFC", "GS", "MA","XOM", "COP", "SLB","CAT", "DE", "UNP", "UPS","NEE", "DUK","LIN"]

select!(eigenvectors_df, :Stock, :)

eigenvectors_df

Row,Stock,Eigenvector 1,Eigenvector 2,Eigenvector 3,Eigenvector 4,Eigenvector 5,Eigenvector 6,Eigenvector 7,Eigenvector 8,Eigenvector 9,Eigenvector 10,Eigenvector 11,Eigenvector 12,Eigenvector 13,Eigenvector 14,Eigenvector 15,Eigenvector 16,Eigenvector 17,Eigenvector 18,Eigenvector 19,Eigenvector 20,Eigenvector 21,Eigenvector 22,Eigenvector 23,Eigenvector 24,Eigenvector 25,Eigenvector 26,Eigenvector 27,Eigenvector 28,Eigenvector 29,Eigenvector 30,Eigenvector 31,Eigenvector 32,Eigenvector 33,Eigenvector 34,Eigenvector 35,Eigenvector 36,Eigenvector 37,Eigenvector 38,Eigenvector 39,Eigenvector 40,Eigenvector 41,Eigenvector 42,Eigenvector 43,Eigenvector 44,Eigenvector 45,Eigenvector 46,Eigenvector 47,Eigenvector 48,Eigenvector 49,Eigenvector 50
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,AAPL,-0.0194734,-0.00800787,-0.0445998,-0.0372393,-0.0499481,0.0315822,-0.141095,0.0577704,-0.0510816,-0.147129,0.00643057,-0.148044,0.0056556,-0.0349313,0.181728,-0.271124,-0.0772435,0.532412,-0.14441,-0.311302,0.0754414,-0.259139,-0.215797,0.166075,-0.145382,0.116149,-0.0612426,0.195031,-0.146606,-0.0685193,-0.0336453,0.0756451,-0.068417,-0.121751,-0.00523921,0.222327,0.106812,0.0518862,-0.00697093,0.0324259,-0.0163596,0.0216171,-0.0252717,-0.0290909,-0.0104594,-0.0800462,0.0107955,-0.148272,0.107682,-0.15876
2,TSLA,0.0182309,-0.00723769,0.0255509,0.00375442,0.0262253,-0.00107884,3.41173e-5,-0.00186625,0.00030506,0.0521959,0.0132551,-0.0287349,-0.00274488,0.0490847,-0.033039,0.0686145,-0.0417549,-0.133033,0.0312411,-0.0470094,-0.0680274,0.134707,0.099411,-0.0791574,0.0238526,-0.0979611,0.0701039,0.112286,-0.0445461,-0.0632604,0.143345,0.0270518,-0.0811025,-0.210888,-0.456793,0.325994,-0.120547,-0.0197522,-0.229121,-0.0901331,-0.568933,-0.168692,-0.0754565,0.0471569,0.0507802,-0.0833271,0.0768892,-0.0986059,0.18248,-0.098843
3,MSFT,-0.014846,0.00845216,0.106996,0.0585835,0.110091,0.110029,0.482254,0.417349,0.0172717,0.483177,-0.0331246,0.163342,-0.109413,0.147742,0.126943,0.00638447,0.221893,-0.0105739,0.0234402,0.0319101,0.0105946,-0.0782742,-0.0474268,0.0374055,-0.0499052,0.114129,-0.0676215,-0.013997,-0.0608501,0.0483299,-0.0372672,0.0116917,-0.0400322,0.0318927,-0.0778855,0.0928519,0.0808757,0.0222406,0.0789129,0.0799079,0.0819695,0.0600594,0.137151,0.00799179,0.0112351,-0.0828694,-0.0272183,-0.193538,0.120491,-0.166869
4,KO,-0.0427812,0.0396687,0.0168593,0.0882437,-0.123375,-0.198618,-0.279969,0.433391,-0.10096,-0.0475218,0.0513327,0.0430471,-0.132599,0.22098,-0.137832,0.201473,-0.238559,0.035807,0.0426719,0.247856,0.386611,0.0282588,-0.0170517,0.0849086,-0.0401927,0.0915016,0.109169,-0.0415638,0.0119869,0.0346386,0.0553987,0.160412,-0.00756468,-0.0115658,0.072532,0.129531,-0.0990782,0.0415334,-0.134383,0.0156588,0.0422434,0.00931821,0.0967858,0.0630046,0.0750853,-0.0490747,0.167853,-0.00849904,-0.260111,-0.143087
5,JNJ,0.00275714,0.0182198,0.0333959,0.00727676,-0.050143,0.0126815,-0.0112534,0.117473,-0.00895794,-0.0579096,0.114942,0.0641551,0.0157548,-0.252515,-0.146667,0.0361053,0.183765,0.140403,-0.303959,0.0962322,-0.0836308,0.289356,0.398918,-0.170755,-0.297418,0.0871557,-0.205713,0.0162577,-0.22162,-0.246787,-0.145566,-0.0172547,-0.056852,-0.132133,0.0444945,0.000452621,-0.00763307,0.043532,-0.0284989,0.0448989,-0.0122744,-0.0516439,0.0622278,-0.0548515,0.00815067,0.00605522,-0.235181,-0.0279065,-0.245165,-0.128699
6,NFLX,0.0114523,-0.0105489,-0.00663501,0.00220153,-0.00336304,0.01085,0.0477913,0.0162443,-0.0384488,0.00696119,-0.000158649,0.00131919,0.0786167,0.0184398,-0.043807,0.0689059,0.027592,0.0624217,0.0717141,-0.0660973,0.0651067,-0.105713,0.021428,-0.0564867,0.0054004

In [94]:
println(minimum(eigvals(Symmetric(cov_julia))) >= -1e-10)
println(minimum(eigvals(Symmetric(corr_julia))) >= -1e-10)

true
true


In [95]:
println(all(corr_julia .>= -1))
println(all(corr_julia .<= 1))

true
true


## Visuals

In [96]:
p = PlotlyLight.Plot()
p.data = [
    PlotlyLight.Config(
        type="heatmap",
        z=corr_julia,
        x=stocks,
        y=stocks,
        colorscale=[[0, "red"], [0.5, "white"], [1, "purple"]],
        zmin=-1,
        zmax=1,
        text=round.(corr_julia, digits=3),
        texttemplate="%{text}"
    )
]
p.layout.title = "Correlation Matrix of 50 stocks"
p.layout.yaxis.autorange = "reversed"
p   

PlotlyLight.jl Plot

In [97]:
writedlm("julia_cov_50.csv", cov_julia, ',')
writedlm("julia_corr_50.csv", corr_julia, ',')